In [1]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

In [2]:
model_name = "microsoft/Phi-4-mini-instruct"

# Select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model onto GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda" if torch.cuda.is_available() else None,
)

# If no GPU is available, move model to CPU
if not torch.cuda.is_available():
    model.to(device)

print("Model device:", next(model.parameters()).device)

Using device: cuda


[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

Model device: cuda:0


In [3]:
phrase = input("Write a phrase: ")
inputs = tokenizer(phrase, return_tensors = "pt")
inputs = {k: v.to(device) for k, v in inputs.items()}
inputs

Write a phrase: Hello World


{'input_ids': tensor([[13225,  5922]], device='cuda:0'),
 'attention_mask': tensor([[1, 1]], device='cuda:0')}

In [4]:
outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

Hello World! I am a 25-year-old male, and I am looking for a job in the field of software development. I have a bachelor's degree in computer science and have worked as a junior developer for the past two years. I am proficient in Java, Python, and C++, and I have experience with web development frameworks such as Django and Flask. I am also familiar with cloud computing platforms such as AWS and Azure. I am looking for a job that allows me to work on challenging projects and grow my skills. I am open to remote work opportunities and am willing to relocate if necessary. I am also interested in companies that prioritize diversity and inclusion. I am available to start as soon as possible and am willing to relocate if necessary. I am looking for a job that offers competitive salary and benefits. I am also interested in opportunities for professional development and career advancement. I am a highly motivated and dedicated individual who is passionate about software development. I am conf

In [15]:


# 1. Assuming you already loaded your model and tokenizer
# model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-4-mini-instruct")
# tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")

# 2. Create a Hugging Face text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    return_full_text=False
)

# 3. Wrap it for LangChain
llm = HuggingFacePipeline(pipeline=pipe)

# 4. Wrap it AGAIN to enable Chat and Tool Calling formats
chat_model = ChatHuggingFace(llm=llm)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [16]:
from langchain_core.tools import tool

@tool
def search_local_database(query: str) -> str:
    """Use this tool to search the local vector database for context."""
    # (Your ChromaDB or FAISS retrieval code will go here later)
    return f"Retrieved documents about: {query}"

# Bind the tool to your Chat Model so Phi-4 knows it exists
model_with_tools = chat_model.bind_tools([search_local_database])

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Step 1: Create some fake company data
fake_company_docs = [
    Document(
        page_content="OmniCorp Vacation Policy: Employees receive 20 days of paid time off (PTO) annually. PTO requests must be submitted through the HR portal at least 2 weeks in advance. Unused PTO does not roll over to the next calendar year.",
        metadata={"source": "hr_vacation_policy.txt"}
    ),
    Document(
        page_content="OmniCorp IT Equipment Guide: All remote employees are eligible for a $500 home office stipend. Company laptops must use the global corporate VPN at all times. Password changes are mandatory every 90 days.",
        metadata={"source": "it_guidelines.txt"}
    ),
    Document(
        page_content="OmniCorp Remote Work Guidelines: Core working hours are 10:00 AM to 4:00 PM EST. Employees working from home must ensure a quiet environment and a minimum internet speed of 50 Mbps down / 10 Mbps up for video conferencing.",
        metadata={"source": "remote_work_policy.txt"}
    )
]

# Step 2: Split the documents into smaller chunks
# (Even though these are short, splitting is a standard habit for RAG pipelines)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = text_splitter.split_documents(fake_company_docs)

# Step 3: Initialize your Embedding Model (all-MiniLM-L6-v2)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Step 4: Create ChromaDB and save it locally
# This creates a folder named 'chroma_db_storage' in your current directory
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db_storage"
)

print(f"Successfully embedded and saved {len(chunks)} text chunks to ChromaDB!")

/tmp/ipykernel_22124/2241893909.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_22124/2241893909.py:28: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Successfully embedded and saved 6 text chunks to ChromaDB!


In [8]:
# Create a retriever object
retriever = vector_store.as_retriever(search_kwargs={"k": 2}) # Get top 2 most relevant chunks

# Run a test search
query = "What is the rule about company laptops and VPN?"
results = retriever.invoke(query)

# Print the results
print(f"Query: {query}\n" + "="*40)
for i, doc in enumerate(results):
    print(f"Result {i+1} from {doc.metadata['source']}:")
    print(f"{doc.page_content}\n")

Query: What is the rule about company laptops and VPN?
Result 1 from it_guidelines.txt:
OmniCorp IT Equipment Guide: All remote employees are eligible for a $500 home office stipend. Company laptops must use the global corporate VPN at all times. Password changes are mandatory every 90

Result 2 from remote_work_policy.txt:
OmniCorp Remote Work Guidelines: Core working hours are 10:00 AM to 4:00 PM EST. Employees working from home must ensure a quiet environment and a minimum internet speed of 50 Mbps down / 10 Mbps up

